# JFP LoRA Fine-tuning — Qwen2.5-1.5B na CPU
Minimalny notebook do treningu bez GPU/Colab.

In [ ]:
# Sprawdzenie środowiska
import torch
print('PyTorch:', torch.__version__)
print('CUDA dostępna:', torch.cuda.is_available())
print('Urządzenie: CPU (wymuszony)')

In [ ]:
import os, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME  = 'Qwen/Qwen2.5-1.5B'
OUTPUT_DIR  = os.path.expanduser('~/Jjfp-core-v1/jfp-lora-adapter')
MAX_LENGTH  = 256
EPOCHS      = 1
BATCH_SIZE  = 1

In [ ]:
# Dane treningowe
EXAMPLES = [
    {'text': '### Instrukcja:\nCzym jest JFP?\n\n### Odpowiedź:\nJFP to projekt fine-tuningu modelu językowego na CPU.'},
    {'text': '### Instrukcja:\nJak działa LoRA?\n\n### Odpowiedź:\nLoRA dodaje małe macierze adaptacyjne, redukując liczbę trenowanych parametrów.'},
    {'text': '### Instrukcja:\nCo to Qwen2.5?\n\n### Odpowiedź:\nQwen2.5 to rodzina modeli od Alibaba Cloud, dostępna m.in. w rozmiarze 1.5B.'},
    {'text': '### Instrukcja:\nJak trenować na CPU?\n\n### Odpowiedź:\nTrening na CPU jest możliwy przy małych modelach i zbiorach danych z LoRA.'},
]
print(f'Przykładów: {len(EXAMPLES)}')

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer załadowany')

In [ ]:
# Model (CPU, float32)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
    device_map='cpu', trust_remote_code=True,
)
model.config.use_cache = False
print('Model załadowany')

In [ ]:
# LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Dataset
ds = Dataset.from_list(EXAMPLES)
def tok(ex):
    out = tokenizer(ex['text'], truncation=True, max_length=MAX_LENGTH, padding='max_length')
    out['labels'] = out['input_ids'][:]
    return out
ds = ds.map(tok, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print('Dataset gotowy')

In [ ]:
# Trening
args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=False, bf16=False,
    logging_steps=1, save_strategy='epoch',
    report_to='none', use_cpu=True, dataloader_num_workers=0,
)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()

In [ ]:
# Zapis
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Adapter zapisany: {OUTPUT_DIR}')